# Data Cleaning

In [43]:
# Load Data Sets

import pandas as pd

car_ownership = pd.read_csv("../data/car_ownership.csv")
median_income = pd.read_csv("../data/median_income.csv")
insurance = pd.read_csv("../data/motor_insurance.csv")
oil_price = pd.read_csv("../data/oil_price.csv")
interest_rates = pd.read_csv("../data/interest_rates.csv")

## Part 1: Median Income

In [16]:
# Convert to date format
median_income['observation_date'] = pd.to_datetime(median_income['observation_date'])

# Extract year
median_income['year'] = median_income['observation_date'].dt.year

# Rename column
median_income = median_income.rename(columns={'MEFAINUSA672N': 'median_income'})

print(median_income)

   observation_date  median_income  year
0        1960-01-01          48760  1960
1        1961-01-01          49270  1961
2        1962-01-01          50670  1962
3        1963-01-01          52400  1963
4        1964-01-01          54550  1964
..              ...            ...   ...
60       2020-01-01         101200  2020
61       2021-01-01         101700  2021
62       2022-01-01          98870  2022
63       2023-01-01         103400  2023
64       2024-01-01         105800  2024

[65 rows x 3 columns]


In [17]:
median_income['difference'] = median_income['median_income'].diff()
median_income.loc[0, 'difference'] = 0
print(median_income)

   observation_date  median_income  year  difference
0        1960-01-01          48760  1960         0.0
1        1961-01-01          49270  1961       510.0
2        1962-01-01          50670  1962      1400.0
3        1963-01-01          52400  1963      1730.0
4        1964-01-01          54550  1964      2150.0
..              ...            ...   ...         ...
60       2020-01-01         101200  2020     -3000.0
61       2021-01-01         101700  2021       500.0
62       2022-01-01          98870  2022     -2830.0
63       2023-01-01         103400  2023      4530.0
64       2024-01-01         105800  2024      2400.0

[65 rows x 4 columns]


In [18]:
# Export copy
median_income.to_csv('../data/new_income.csv', index=False)

## Part 2: Oil Prices

In [10]:
# Convert to date format
oil_price['Date'] = pd.to_datetime(oil_price['Date'])

# Filter
oil_price = oil_price[(oil_price['Date'].dt.year != 2025) & (oil_price['Date'].dt.year != 2026)]

# Extract Year
oil_price['year'] = oil_price['Date'].dt.year

# Extract Month
oil_price['month'] = oil_price['Date'].dt.strftime('%b')

# Export copy
oil_price.to_csv('../data/new_oil.csv', index=False)

In [ ]:
# Filter by January and only until 2024
oil_filtered = oil_price[oil_price['Date'].dt.month == 3]

# Rename column
oil_filtered = oil_filtered.rename(columns={'Value': 'oil_price'})

print(oil_filtered)

          Date  oil_price  year
2   1960-03-01      33.36  1960
14  1961-03-01      32.91  1961
26  1962-03-01      32.58  1962
38  1963-03-01      32.16  1963
50  1964-03-01      31.74  1964
..         ...        ...   ...
722 2020-03-01      26.24  2020
734 2021-03-01      73.79  2021
746 2022-03-01     115.46  2022
758 2023-03-01      82.80  2023
770 2024-03-01      88.77  2024

[65 rows x 3 columns]


## Part 3: Insurance

In [3]:
# Only grab one month
new_insurance = insurance[['Year', 'Jun']]

# Filter years
new_insurance = new_insurance[(new_insurance['Year'] != 2025) & (new_insurance['Year']!= 2026)]

# Rename columns
new_insurance = new_insurance.rename(columns={'Year': 'year', 'Jun': 'insurance'})

print(new_insurance)

    year  insurance
0   1960     25.700
1   1961     26.000
2   1962     26.100
3   1963     26.200
4   1964     27.100
..   ...        ...
60  2020    511.639
61  2021    569.656
62  2022    603.932
63  2023    705.717
64  2024    843.579

[65 rows x 2 columns]


In [4]:
insurance_clean = insurance.drop(columns = ['HALF1', 'HALF2'])

In [5]:
insurance_clean = insurance_clean.melt(
    id_vars=['Year'],     
    var_name='month',        
    value_name='insurance' 
)

insurance_clean['date'] = pd.to_datetime(insurance_clean['Year'].astype(str) + '-' + insurance_clean['month'] + '-01', format='%Y-%b-%d')

In [6]:
# Filter years
final_insurance = insurance_clean[(insurance_clean['Year'] != 2025) & (insurance_clean['Year']!= 2026)]

final_insurance = final_insurance.rename(columns={'Year': 'year'})

print(final_insurance)

     year month  insurance       date
0    1960   Jan        NaN 1960-01-01
1    1961   Jan        NaN 1961-01-01
2    1962   Jan        NaN 1962-01-01
3    1963   Jan        NaN 1963-01-01
4    1964   Jan        NaN 1964-01-01
..    ...   ...        ...        ...
797  2020   Dec    545.376 2020-12-01
798  2021   Dec    567.875 2021-12-01
799  2022   Dec    648.771 2022-12-01
800  2023   Dec    780.284 2023-12-01
801  2024   Dec    868.417 2024-12-01

[780 rows x 4 columns]


In [7]:
final_insurance.to_csv('../data/new_insurance.csv', index=False)

# Part 4: Car Ownership

In [ ]:
new_car = car_ownership.melt(
    id_vars=['cars'],     
    var_name='year',       
    value_name='ownership_perc'      
)

new_car['ownership_perc'] = new_car['ownership_perc'].str.replace('%', '', regex=False)
new_car['ownership_perc'] = new_car['ownership_perc'].astype(float)
new_car['year'] = new_car['year'].astype(int)

new_car.to_csv('../data/new_car.csv', index=False)

# Combine Columns

In [39]:
df = pd.merge(new_insurance, oil_filtered, on='year')
df = pd.merge(df, median_income, on='year')

df_final = df.drop(columns= ['Date', 'observation_date'])

print(df_final)

    year  insurance  oil_price  median_income
0   1960     25.700      33.36          48760
1   1961     26.000      32.91          49270
2   1962     26.100      32.58          50670
3   1963     26.200      32.16          52400
4   1964     27.100      31.74          54550
..   ...        ...        ...            ...
60  2020    511.639      26.24         101200
61  2021    569.656      73.79         101700
62  2022    603.932     115.46          98870
63  2023    705.717      82.80         103400
64  2024    843.579      88.77         105800

[65 rows x 4 columns]


In [40]:
df_final.to_csv('../data/df_final.csv', index=False)

# Part 5: Interest Rates

In [44]:
# Convert to date format
interest_rates['Date'] = pd.to_datetime(interest_rates['Date'])

In [45]:
# Extract Year
interest_rates['year'] = interest_rates['Date'].dt.year

# Filter years
new_interest = interest_rates[(interest_rates['Date'] == '1960-01-01') | (interest_rates['Date']== '1970-01-01') | (interest_rates['Date']== '1980-01-01') | (interest_rates['Date']== '1990-01-01') | (interest_rates['Date']== '2000-01-01') | (interest_rates['Date']== '2010-01-01') | (interest_rates['Date']== '2021-01-01') | (interest_rates['Date']== '2022-01-01') | (interest_rates['Date']== '2023-01-01')]

print(new_interest)

          Date         Value  year
68  1960-01-01  1.817224e+11  1960
108 1970-01-01  2.999658e+11  1970
148 1980-01-01  4.534422e+11  1980
188 1990-01-01  7.086591e+11  1990
228 2000-01-01  9.679619e+11  2000
268 2010-01-01  1.044007e+12  2010
312 2021-01-01  1.487955e+12  2021
316 2022-01-01  1.566766e+12  2022
320 2023-01-01  1.591909e+12  2023


In [46]:
new_interest.to_csv('../data/new_interest.csv', index=False)